In [201]:
import math
import random
import numpy
import torch
import torch.nn as nn
from torch.nn import functional as F

In [238]:
batch_size = 32
block_size = 128
n_embd = 128
n_head = 4
epochs = 100000
learning_rate = 3e-4
n_layers = 4
dropout = 0.2
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f"using device: {device}")

using device: mps


In [223]:
with open('data/grabbed_nietzsche.txt', 'r', encoding='utf-8') as f:
    nietzshe = f.read()

In [224]:
len(nietzshe)

5812740

In [225]:
characters = sorted(set(nietzshe))
vocab_size = len(characters)
print(''.join(characters))
print(vocab_size)


 !"&'()*,-./0123456789:;<=>?ABCDEFGHIJKLMNOPQRSTUVWXYZ[\]_abcdefghijklmnopqrstuvwxyz}§ÀÄÆÈÉÊÏÜàâäæçèéêîïôöùûüŒœΚΣΤάέήίαβγδεζηθικλμνξοπρςστυφχψωόύώἀἃἄἆἈἐἑἒἔἰὀὖὰὲὶὸᾶ᾽ῑῡῢῶ‐–—‘’“”…
177


In [226]:
stoi = {s:i for i, s in enumerate(characters)}
itos = {i:s for s, i in stoi.items()}
print(stoi)
print(itos)                                     # the encoders we are using are character to token encoders

encode = lambda s: [stoi[c] for c in s]
decode = lambda s: ''.join([itos[c] for c in s])
encode('hi there')
decode(encode("hi there"))


{'\n': 0, ' ': 1, '!': 2, '"': 3, '&': 4, "'": 5, '(': 6, ')': 7, '*': 8, ',': 9, '-': 10, '.': 11, '/': 12, '0': 13, '1': 14, '2': 15, '3': 16, '4': 17, '5': 18, '6': 19, '7': 20, '8': 21, '9': 22, ':': 23, ';': 24, '<': 25, '=': 26, '>': 27, '?': 28, 'A': 29, 'B': 30, 'C': 31, 'D': 32, 'E': 33, 'F': 34, 'G': 35, 'H': 36, 'I': 37, 'J': 38, 'K': 39, 'L': 40, 'M': 41, 'N': 42, 'O': 43, 'P': 44, 'Q': 45, 'R': 46, 'S': 47, 'T': 48, 'U': 49, 'V': 50, 'W': 51, 'X': 52, 'Y': 53, 'Z': 54, '[': 55, '\\': 56, ']': 57, '_': 58, 'a': 59, 'b': 60, 'c': 61, 'd': 62, 'e': 63, 'f': 64, 'g': 65, 'h': 66, 'i': 67, 'j': 68, 'k': 69, 'l': 70, 'm': 71, 'n': 72, 'o': 73, 'p': 74, 'q': 75, 'r': 76, 's': 77, 't': 78, 'u': 79, 'v': 80, 'w': 81, 'x': 82, 'y': 83, 'z': 84, '}': 85, '§': 86, 'À': 87, 'Ä': 88, 'Æ': 89, 'È': 90, 'É': 91, 'Ê': 92, 'Ï': 93, 'Ü': 94, 'à': 95, 'â': 96, 'ä': 97, 'æ': 98, 'ç': 99, 'è': 100, 'é': 101, 'ê': 102, 'î': 103, 'ï': 104, 'ô': 105, 'ö': 106, 'ù': 107, 'û': 108, 'ü': 109, 'Œ': 11

'hi there'

In [227]:
tokenized_nietzshe = torch.tensor(encode(nietzshe), dtype=torch.long)       # encode the entirety of nietzshe into tokens
                                                                            

tokenized_nietzshe[:1000]

tensor([ 48,  36,  33,   1,  48,  51,  37,  40,  37,  35,  36,  48,   1,  43,
         34,   1,  48,  36,  33,   1,  37,  32,  43,  40,  47,   0,   0,  30,
         53,   0,   0,  34,  46,  37,  33,  32,  46,  37,  31,  36,   1,  42,
         37,  33,  48,  54,  47,  31,  36,  33,   0,   0,  43,  76,   9,   1,
         36,  73,  81,   1,  78,  73,   1,  44,  66,  67,  70,  73,  77,  73,
         74,  66,  67,  77,  63,   1,  81,  67,  78,  66,   1,  78,  66,  63,
          1,  36,  59,  71,  71,  63,  76,   0,   0,   0,  48,  36,  33,   1,
         29,  42,  48,  37,  31,  36,  46,  37,  47,  48,   0,   0,  58,  42,
         43,  48,  33,  47,   1,  48,  43,   1,  54,  29,  46,  29,  48,  36,
         49,  47,  48,  46,  29,   9,   1,  29,  42,  32,   1,  33,  48,  33,
         46,  42,  29,  40,   1,  46,  33,  31,  49,  46,  46,  33,  42,  31,
         33,  58,   0,   0,   0,  48,  46,  29,  42,  47,  40,  29,  48,  33,
         32,   1,  30,  53,   0,   0,  29,  42,  48,  36,  43,  

In [228]:
# split into train and test data
n = int(len(tokenized_nietzshe)*.9)
train_nietzshe = tokenized_nietzshe[:n]
test_nietzshe = tokenized_nietzshe[n:]

print(train_nietzshe.shape, test_nietzshe.shape)

torch.Size([5231466]) torch.Size([581274])


In [229]:
train_nietzshe[:block_size+1]

tensor([48, 36, 33,  1, 48, 51, 37, 40, 37, 35, 36, 48,  1, 43, 34,  1, 48, 36,
        33,  1, 37, 32, 43, 40, 47,  0,  0, 30, 53,  0,  0, 34, 46, 37, 33, 32,
        46, 37, 31, 36,  1, 42, 37, 33, 48, 54, 47, 31, 36, 33,  0,  0, 43, 76,
         9,  1, 36, 73, 81,  1, 78, 73,  1, 44, 66, 67, 70, 73, 77, 73, 74, 66,
        67, 77, 63,  1, 81, 67, 78, 66,  1, 78, 66, 63,  1, 36, 59, 71, 71, 63,
        76,  0,  0,  0, 48, 36, 33,  1, 29, 42, 48, 37, 31, 36, 46, 37, 47, 48,
         0,  0, 58, 42, 43, 48, 33, 47,  1, 48, 43,  1, 54, 29, 46, 29, 48, 36,
        49, 47, 48, 46, 29,  9,  1, 29, 42, 32,  1, 33, 48, 33, 46, 42, 29, 40,
         1, 46, 33, 31, 49, 46, 46, 33, 42, 31, 33, 58,  0,  0,  0, 48, 46, 29,
        42, 47, 40, 29, 48, 33, 32,  1, 30, 53,  0,  0, 29, 42, 48, 36, 43, 42,
        53,  1, 41, 11,  1, 40, 49, 32, 43, 50, 37, 31, 37,  0,  0,  0, 48, 66,
        63,  1, 31, 73, 71, 74, 70, 63, 78, 63,  1, 51, 73, 76, 69, 77,  1, 73,
        64,  1, 34, 76, 67, 63, 62, 76, 

In [230]:
v = torch.randint(n-block_size, (batch_size, ))     # how to sample random values from the trainset
for i in v:
    print(i.item())

2052516
1119029
1005914
3434571
4261412
2353023
3213091
1175116
337298
4759472
5177322
501204
4825829
4935341
2503591
2228104
1563326
2285908
1906319
4588058
4854145
2502122
4965736
694810
4707817
3259989
2996576
3559424
3386417
2694456
219850
434789
1413488
2436162
2698676
637239
568143
1809888
3494901
1179988
4448429
670919
5133040
2796700
4238316
3952470
1060842
3777798
3492413
4749689
3511605
2333731
5067006
3523282
5120580
998236
754422
773796
1581058
2226713
1497883
691158
3885342
4585556


In [231]:

def generate_batch(type=None):
    xb = []
    yb = []

    if type == 'test':
        data = test_nietzshe
    else:
        data = train_nietzshe

    n = len(data)
    start_vals = torch.randint(n-block_size, (batch_size, ))     # how to sample random values from the trainset 
    for start_val in start_vals:
        start_val = start_val.item()
        xb.append(data[start_val:start_val+block_size])
        yb.append(data[start_val+1:start_val+block_size+1])
    
    xb = torch.stack(xb)
    yb = torch.stack(yb)
    return xb.to(device), yb.to(device)                          # move batches onto the active device

xb, yb = generate_batch()

print(xb, yb)

tensor([[79, 72, 64,  ...,  1, 77, 73],
        [63, 63, 72,  ...,  1, 63, 67],
        [64,  1, 73,  ..., 66, 67, 77],
        ...,
        [79, 78,  1,  ..., 74, 63, 76],
        [65, 72, 67,  ..., 63, 59, 78],
        [67, 78, 83,  ..., 78,  1, 59]], device='mps:0') tensor([[72, 64, 59,  ..., 77, 73, 10],
        [63, 72,  1,  ..., 63, 67, 65],
        [ 1, 73, 70,  ..., 67, 77,  1],
        ...,
        [78,  1, 78,  ..., 63, 76, 67],
        [72, 67, 64,  ..., 59, 78, 63],
        [78, 83, 11,  ...,  1, 59,  1]], device='mps:0')


In [232]:
class SelfAttention(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size)
        self.query = nn.Linear(n_embd, head_size)
        self.value = nn.Linear(n_embd, head_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, xb):                                              # (B, T, C)
        k = self.key(xb)                                                # (B, T, head_size=n_embd/n_head)
        q = self.query(xb)                                              # (B, T, head_size)
        v = self.value(xb)                                              # (B, T, head_size)

        B, T, C = k.shape

        weights = q @ k.transpose(-2, -1) / C**0.5                      # (B, T, C) @ (B, C, T) => (B, T, T)
                                                                        # divided by n_embd root to normalize the values, bring them to one
        mask = torch.tril(torch.ones(T, T, device=xb.device))          # triangular mask, built on the same device as the input
        weights = weights.masked_fill(mask == 0, float('-inf'))      
        weights = F.softmax(weights, dim=-1)                            # softmax to get probabilities (B, T, T)
        weights = self.dropout(weights)

        out = weights @ v                                               # (B, T, T) @ (B, T, head_size) => (B, T, head_size)
        return out
      

In [233]:
class MultipleSelfAttention(nn.Module):
    def __init__(self, n_head, n_embd):
        super().__init__()
        self.heads = nn.ModuleList([SelfAttention(n_embd // n_head) for i in range(n_head)])
        self.lin_proj = nn.Linear(n_embd, n_embd)

    def forward(self, xb):
        full_attention = []

        for head in self.heads: 
            full_attention.append(head(xb))                             # (B, T, head_size)
        
        full_attention = torch.cat(full_attention, dim=-1)              # add n_head of (B, T, head_size) along last dimension = (B, T, C)
        full_attention = self.lin_proj(full_attention)                  # linear projection of (B, T, C) to get (B, T, C)

        return full_attention


In [234]:
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.ff = nn.Sequential(
            nn.Linear(n_embd, 4*n_embd),
            nn.GELU(),
            nn.Linear(4*n_embd, n_embd),
            nn.Dropout(dropout)
        )
    
    def forward(self, xb):
        return self.ff(xb)

In [235]:
class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.full_attend = MultipleSelfAttention(n_head, n_embd)
        self.ff = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, xb):
        attention = xb + self.full_attend(self.ln1(xb))                      # (B, T, C) => 4 * self_attend(B, T, C/4) => (B, T, C)
        feed_forward = attention + self.ff(self.ln2(attention))

        return feed_forward

In [239]:
class LTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embd)         # a token embedding allows us to represent letters as vectors
        self.pos_embedding = nn.Embedding(block_size, n_embd)           # attention also depends on the location of a token
                                                                        # the token embedding itself will give the same value to a character at the front vs at the end
                                                                        # pos embedding handles this
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for i in range(n_layers)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.last_layer = nn.Linear(n_embd, vocab_size)

    def forward(self, xb, targets):
        B, T = xb.shape

        tok_embd = self.token_embedding(xb)                             # (B, T, C)
        pos_embd = self.pos_embedding(torch.arange(T, device=xb.device))  # (T, C), positions built on the input's device
        x = tok_embd + pos_embd                                       # (B, T, C) + (T, C) => (B, T, C) + (B, T, C)

        out = self.blocks(x)
        logits = self.last_layer(self.ln_f(out))
        
        if targets is None: 
            loss = None
        else:
            flat_logits = logits.view(-1, vocab_size)
            flat_targets = targets.view(-1)
            loss = F.cross_entropy(flat_logits, flat_targets)
        return logits, loss
    
    def generate(self, tokens, idx):

        for i in range(tokens):
            logits, loss = self(idx[:, -block_size:], None)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            next_tok = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_tok), dim=1)
        
        return idx


xb, yb = generate_batch()

model = LTransformer().to(device)                                   # move all model parameters onto the active device
logits, loss = model(xb, yb)
print(logits, loss)

tensor([[[ 0.6344,  0.2115,  0.5374,  ..., -0.0500,  0.5184,  0.2437],
         [ 0.1884,  0.1141,  0.1616,  ...,  0.4117,  0.1438,  0.1392],
         [-0.3216, -0.7730, -0.0893,  ..., -0.2658,  0.0266, -0.1628],
         ...,
         [ 0.0154,  0.1269,  0.0830,  ...,  0.2527, -0.2841, -0.2556],
         [ 1.1083,  0.1189, -0.4449,  ..., -0.6537,  0.4292, -0.7140],
         [ 0.0075,  0.1896, -0.1272,  ...,  0.3592, -0.1372,  0.1860]],

        [[ 0.6533,  0.1250, -0.3963,  ...,  0.7209,  0.2198,  0.8349],
         [ 1.1166,  0.7638,  0.2706,  ...,  0.1258,  1.0984, -0.2228],
         [ 0.5232,  0.1203,  0.6399,  ...,  0.6617,  0.1487,  0.2235],
         ...,
         [ 1.0824,  0.5215,  0.3769,  ..., -0.3001,  0.3375, -0.7001],
         [ 1.3487,  0.6979,  0.1595,  ...,  0.3878,  0.6230, -0.6063],
         [-0.0252,  0.2590, -0.2837,  ...,  1.1154,  0.4951,  0.4436]],

        [[ 0.8200, -0.2136,  1.1372,  ...,  0.5591,  0.8473,  0.4049],
         [ 0.9079,  0.5587,  0.7033,  ..., -0

In [240]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

@torch.no_grad()
def estimate_loss(iters):
    out = {}
    model.eval()
    for split in ['train', 'test']:
        losses = torch.zeros(iters)
        for k in range(iters):
            xb, yb = generate_batch(split)                  # use the split's data (train vs test)
            tlogits, tloss = model(xb, yb)
            losses[k] = tloss.item()
        out[split] = losses.mean()
    model.train()
    return out

for i in range(epochs):
    xb, yb = generate_batch()
    logits, loss = model(xb, yb)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if i % 1000 ==0:
        estimated_loss = estimate_loss(20)
        print(f"loss at epoch {i}: {estimated_loss['train']:.4f}, {estimated_loss['test']:.4f}")

loss at epoch 0: 5.2293, 5.2348
loss at epoch 1000: 2.2629, 2.2842
loss at epoch 2000: 1.8549, 1.8741
loss at epoch 3000: 1.7090, 1.7164
loss at epoch 4000: 1.6252, 1.6287
loss at epoch 5000: 1.5545, 1.5494
loss at epoch 6000: 1.5268, 1.5076
loss at epoch 7000: 1.4737, 1.4808
loss at epoch 8000: 1.4511, 1.4571
loss at epoch 9000: 1.4414, 1.4180
loss at epoch 10000: 1.4115, 1.4164
loss at epoch 11000: 1.3940, 1.3981
loss at epoch 12000: 1.3911, 1.3902
loss at epoch 13000: 1.3784, 1.3731
loss at epoch 14000: 1.3683, 1.3662
loss at epoch 15000: 1.3568, 1.3535
loss at epoch 16000: 1.3392, 1.3444
loss at epoch 17000: 1.3565, 1.3402
loss at epoch 18000: 1.3328, 1.3362
loss at epoch 19000: 1.3392, 1.3456
loss at epoch 20000: 1.3125, 1.3211
loss at epoch 21000: 1.3199, 1.3231
loss at epoch 22000: 1.3086, 1.3091
loss at epoch 23000: 1.3015, 1.3132
loss at epoch 24000: 1.3009, 1.3034
loss at epoch 25000: 1.2903, 1.3013
loss at epoch 26000: 1.2816, 1.2880
loss at epoch 27000: 1.2807, 1.2941
loss 

In [243]:
idx = torch.zeros((1, 1), dtype=torch.long, device=device)          # seed token, on the active device
out = model.generate(200, idx)
print(decode(out[0].tolist()))


between the essentagences of superful phenomena actions, “i happened
Zarathustra,” in all the functions of “great principle!”





248.



THE HONEST TASTE OF EDYMPLE.—That take cults it compression b
